In [28]:
# Uncomment if needed:
# %pip install -U scipy pandas numpy matplotlib seaborn

from pathlib import Path
import ast, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pingouin as pg

from statannotations.Annotator import Annotator
warnings.filterwarnings("ignore")

### Setting up

In [79]:
DATA_PATH = r"C:\Users\jvill\Downloads\jorge_kidney_data_annotated_yang\jorge_kidney_data_annotated_yang\masked_data.csv"
ROOT_DIR = Path(r"C:\Users\jvill\Downloads\jorge_kidney_data_annotated_yang\jorge_kidney_data_annotated_yang")

ALPHA = 0.05

GROUP_ORDER = ["NC + veh", "NC + dapa", "WD + veh", "WD + dapa"]
GROUP_PALETTE = {
    "NC + veh": "#2596be",
    "NC + dapa": "#76b5c5",
    "WD + veh": "#e28743",
    "WD + dapa": "#eab676",
}

In [80]:
MOLECULE_COLUMNS = [
    "Sterols Normalized",
    "Phospholipids Normalized",
    "Glycerols Normalized",
    "Sphingolipids Normalized",
    "Fatty Acids Normalized",
    "Lipid Droplet Sterols Normalized",
    "Lipid Droplet Phospholipids Normalized",
    "Lipid Droplet Glycerols Normalized",
    "Lipid Droplet Sphingolipids Normalized",
    "Lipid Droplet Fatty Acids Normalized",
]

MAJORITY_COL = "Lipid Droplet Majority Molecules"

MOLECULE_PALETTE = {
    "Sterols": "#CC79A7",
    "Phospholipids": "#009E73",
    "Glycerols": "#F0E442",
    "Sphingolipids": "#0072B2",
    "Fatty Acids": "#D55E00",
}



In [81]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

ROOT_DIR.mkdir(parents=True, exist_ok=True)

print(df.shape)
df.head()

(160, 34)


,Image_Name,Unit,Image Index,Nuclei_count,Group Number,Group,Instance_Label,centroid_x,centroid_y,n_pixels,...,Lipid Droplet Count,Lipid Droplet Majority Molecules,Lipid Droplet Avg Area um2,Lipid Droplet Background,Lipid Droplet Sterols,Lipid Droplet Phospholipids,Lipid Droplet Glycerols,Lipid Droplet Sphingolipids,Lipid Droplet Fatty Acids,Lipid Droplet Carnitines
0,2_1-EarlyPT-1-HSI,EarlyPT,1,9,1,NC + veh,01,262.680879,257.579049,106972,...,144.0,"['Sterols', 'Sterols', 'Sterols', 'Sterols', '...",0.447021,0.0,0.597098,0.312420,0.040546,0.049936,0.000000,0.0
1,2_1-EarlyPT-2-HSI,EarlyPT,1,7,2,NC + veh,01,194.117421,127.596163,60049,...,33.0,"['Sterols', 'Sterols', 'Sterols', 'Sterols', '...",0.414551,0.0,0.570281,0.333333,0.054217,0.040161,0.002008,0.0
2,2_1-EarlyPT-2-HSI,EarlyPT,2,5,3,NC + veh,02,138.477185,381.888575,37541,...,18.0,"['Sterols', 'Sterols', 'Sterols', 'Sterols', '...",0.431885,0.0,0.583039,0.335689,0.070671,0.010601,0.000000,0.0
3,2_1-EarlyPT-2-HSI,EarlyPT,3,6,4,NC + veh,03,374.499955,390.847166,44604,...,18.0,"['Sterols', 'Sterols', 'Sterols', 'Sterols', '...",0.401367,0.0,0.646388,0.285171,0.034221,0.034221,0.000000,0.0
4,2_1-EarlyPT-2-HSI,LatePT,1,5,1,NC + veh,01,340.439334,214.857861,29028,...,24.0,"['Sterols', 'Phospholipids', 'Phospholipids', ...",0.427002,0.0,0.533512,0.380697,0.048257,0.037534,0.000000,0.0


In [ ]:
ts_map = {img : img.split("-")[0] for img in df['Image_Name'].unique()}
df['Tissue_Section'] = df['Image_Name'].map(ts_map)

df['Droplet Count per Cell'] = df['Lipid Droplet Count'].div(df['Nuclei_count']).replace([np.inf, -np.inf], np.nan)

numeric_columns = ['Sterols', 'Phospholipids', 'Glycerols', 'Sphingolipids', 'Fatty Acids', 'Lipid Droplet Count', 'Lipid Droplet Avg Area um2', 'Droplet Count per Cell', 'Lipid Droplet Sterols', 'Lipid Droplet Phospholipids', 'Lipid Droplet Glycerols', 'Lipid Droplet Sphingolipids', 'Lipid Droplet Fatty Acids']

for col in numeric_columns:
    ug_mean_df = df.groupby(['Unit'])[col].mean()
    ug_mean = ug_mean_df.xs('NC + veh', level='Group')
    df[f'{col} Ctrl Mean'] = df['Unit'].map(ug_mean)
    df[f'{col} Normalized'] = df[col] / df[f'{col} Ctrl Mean']
# df_clean = df.dropna()
df.head()

,Image_Name,Unit,Image Index,Nuclei_count,Group Number,Group,Instance_Label,centroid_x,centroid_y,n_pixels,...,Lipid Droplet Sterols Ctrl Mean,Lipid Droplet Sterols Normalized,Lipid Droplet Phospholipids Ctrl Mean,Lipid Droplet Phospholipids Normalized,Lipid Droplet Glycerols Ctrl Mean,Lipid Droplet Glycerols Normalized,Lipid Droplet Sphingolipids Ctrl Mean,Lipid Droplet Sphingolipids Normalized,Lipid Droplet Fatty Acids Ctrl Mean,Lipid Droplet Fatty Acids Normalized
0,2_1-EarlyPT-1-HSI,EarlyPT,1,9,1,NC + veh,01,262.680879,257.579049,106972,...,0.294506,2.027456,0.657146,0.475420,0.017108,2.370023,0.030916,1.615206,0.000324,0.000000
1,2_1-EarlyPT-2-HSI,EarlyPT,1,7,2,NC + veh,01,194.117421,127.596163,60049,...,0.294506,1.936400,0.657146,0.507244,0.017108,3.169098,0.030916,1.299018,0.000324,6.194829
2,2_1-EarlyPT-2-HSI,EarlyPT,2,5,3,NC + veh,02,138.477185,381.888575,37541,...,0.294506,1.979719,0.657146,0.510829,0.017108,4.130901,0.030916,0.342886,0.000324,0.000000
3,2_1-EarlyPT-2-HSI,EarlyPT,3,6,4,NC + veh,03,374.499955,390.847166,44604,...,0.294506,2.194822,0.657146,0.433954,0.017108,2.000267,0.030916,1.106882,0.000324,0.000000
4,2_1-EarlyPT-2-HSI,LatePT,1,5,1,NC + veh,01,340.439334,214.857861,29028,...,0.212420,2.511589,0.738838,0.515265,0.015398,3.133998,0.033041,1.135963,0.000303,0.000000


In [83]:
df = df[df['Tissue_Section']!='15_1']

### Define helper functions

In [84]:
def safe_name(x):
    x = str(x).strip().replace(" ", "_").replace("/", "-")
    return re.sub(r"[^A-Za-z0-9_.+-]", "", x)


def present_order(values, preferred_order=GROUP_ORDER):
    values = pd.Series(values).dropna()
    vals = [v for v in preferred_order if v in set(values)]
    extras = sorted([v for v in values.unique() if v not in vals])
    return vals + extras


def numeric_subset(df, outcome):
    # tmp = df[["Unit", "Group", outcome]].copy()
    tmp = df.copy()
    tmp[outcome] = pd.to_numeric(tmp[outcome], errors="coerce")
    tmp = tmp.dropna(subset=["Unit", "Group", outcome])
    return tmp.rename(columns={outcome: "value"})

In [85]:
def scipy_anova_group_unit_combo(df, outcome):
    tmp = numeric_subset(df, outcome)
    tmp["Group_Unit"] = tmp["Group"].astype(str) + " | " + tmp["Unit"].astype(str)

    groups = []

    for label, sub in tmp.groupby("Group_Unit"):
        vals = sub["value"].dropna().values
        if len(vals) >= 2:
            groups.append((label, vals))
        else:
            print(f'Not enough data points for {label}')

    if len(groups) < 2:
        return pd.DataFrame([{
            "outcome": outcome,
            "factor": "Group_Unit",
            "F": np.nan,
            "p_value": np.nan,
            "n_groups": len(groups),
            "note": "Not enough valid Group-Unit groups"
        }])

    res = stats.f_oneway(*[g[1] for g in groups])

    return pd.DataFrame([{
        "outcome": outcome,
        "factor": "Group_Unit",
        "F": res.statistic,
        "p_value": res.pvalue,
        "n_groups": len(groups),
        "significant": res.pvalue < ALPHA
    }])

In [86]:
def scipy_anova_group_within_unit(df, outcome, unit):
    tmp = numeric_subset(df, outcome)
    tmp = tmp[tmp["Unit"] == unit]

    groups = []

    for label, sub in tmp.groupby("Group"):
        vals = sub["value"].dropna().values
        if len(vals) >= 2:
            groups.append((label, vals))
        else:
            print(f'Not enough data points for {label}: {len(vals)}')

    if len(groups) < 2:
        return np.nan, np.nan, len(groups)

    res = stats.f_oneway(*[g[1] for g in groups])


    return res.statistic, res.pvalue, len(groups)

In [87]:
def scipy_posthoc_by_unit(df, outcome, unit, equal_var=True):
    tmp = numeric_subset(df, outcome)
    tmp = tmp[tmp["Unit"] == unit]

    labels, arrays = [], []

    for label, sub in tmp.groupby("Group"):
        vals = sub["value"].dropna().values
        if len(vals) >= 2:
            labels.append(label)
            arrays.append(vals)

    if len(arrays) < 2:
        return pd.DataFrame([{
            "outcome": outcome,
            "unit": unit,
            "note": "Not enough valid groups for post-hoc test"
        }])

    res = stats.tukey_hsd(*arrays, equal_var=equal_var)
    ci = res.confidence_interval(confidence_level=0.95)

    rows = []

    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            rows.append({
                "outcome": outcome,
                "unit": unit,
                "group1": labels[i],
                "group2": labels[j],
                "test": "Tukey-Kramer" if equal_var else "Games-Howell",
                "statistic": res.statistic[i, j],
                "p_adj": res.pvalue[i, j],
                "ci_low": ci.low[i, j],
                "ci_high": ci.high[i, j],
                "reject": res.pvalue[i, j] < ALPHA,
            })

    return pd.DataFrame(rows)

In [88]:
def remove_outliers_iqr(group, col):
    Q1 = group[col].quantile(0.25)
    Q3 = group[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return group[(group[col] >= lower_bound) & (group[col] <= upper_bound)]

def violin_scatter_plot(df, outcome, unit, out_path, title=None):
    tmp = numeric_subset(df, outcome)
    tmp = tmp[tmp["Unit"] == unit]

    tmp_filtered = tmp.groupby('Group', group_keys=False).apply(remove_outliers_iqr, col='value')
    if tmp_filtered.empty:
        return

    counts = tmp['Group'].value_counts()
    valid_groups = counts[counts >= 2].index
    tmp_filtered = tmp[tmp['Group'].isin(valid_groups)]

    order = present_order(tmp["Group"])

    plt.figure(figsize=(max(6, 1.35 * len(order)), 4.8))
    # ax = sns.violinplot(
    #     data=tmp_filtered, 
    #     x="Group", 
    #     y="value",
    #       order=order,
    #     palette=GROUP_PALETTE,
    #       inner=None, cut=0, 
    #       linewidth=2
    # )
    ax = sns.barplot(
        data=tmp_filtered,
        x='Group',
        y='value',
        errorbar='se',         
        capsize=0.2,      # Adds caps to error bars
        order=order,
        palette=GROUP_PALETTE,
        # inner=None, cut=0, 
        linewidth=2
    )

    sns.stripplot(
        data=tmp, x="Group", y="value", 
        order=order,
        hue="Tissue_Section",          # Color points by another column
        # dodge=True,           # Separate hue groups within categories
        palette="Set2",       # Color palette
        # color="black",          # Color points by another column --- IGNORE ---
        edgecolor="black",
        linewidth=0.2,
        jitter=0.18, size=4, alpha=0.8, ax=ax
    )

    # unit_significance = []
    # sig_symbols = {0.001: '***', 0.01: '**', 0.05: '*', 1.0: 'ns'}
    # # unit_position_map = {} 
    # y_max = tmp['value'].max()
    # offset=0.1

    # 1.  ANOVA
    aov = pg.anova(data=tmp, dv='value', between='Group')
    if not aov.empty and 'p_unc' in aov.columns and aov['p_unc'].iloc[0] < 0.05:
        # 2. Games-Howell Post-hoc
        ph = pg.pairwise_gameshowell(data=tmp, dv='value', between='Group')
        # pairs = list(zip(ph['A'], ph['B']))
        # pvalues = ph['pval'].tolist()
        sig_results = ph[ph['pval'] < 0.05]

        # Extract pairs and p-values
        pairs = list(zip(sig_results['A'], sig_results['B']))
        pvalues = sig_results['pval'].tolist()
        # Filter only significant results (p < 0.05)
        if len(pairs) > 0:
            annotator = Annotator(ax, pairs, data=tmp, x='Group', y='value')
            annotator.set_pvalues(pvalues)
            annotator.configure(test=None, text_format='star', loc='outside', verbose=2)
            annotator.annotate()
        # for _, row in ph.iterrows():
        #     u1, u2, p = row['A'], row['B'], row['pval']
        #     if p < 0.05:
        #         sig = next((s for t, s in sig_symbols.items() if p <= t), 'ns')
        #         # unit_significance.append((unit_position_map[u1], unit_position_map[u2], sig))
        #         ax.plot([u1, u1, u2, u2],
        #                 [y_max + offset, y_max + offset + 0.1, y_max + offset + 0.1, y_max + offset],
        #                 lw=1.5, c='black')
        #         ax.text((u1 + u2) / 2, y_max + offset + 0.15, sig,
        #                 ha='center', va='bottom', color='black')
        #         offset += 0.1  # Increase offset for next annotation
            y_max = tmp['value'].max()
            ax.set_ylim(0, y_max + y_max/2)  # Add extra space above bars





    ax.set_title(title or f"{outcome} | {unit}")
    ax.set_xlabel("Group")
    ax.set_ylabel(outcome)
    ax.tick_params(axis="x", rotation=30)



    legend = plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    legend.remove()
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()

In [89]:
def parse_majority_list(x):
    if pd.isna(x):
        return []

    try:
        vals = ast.literal_eval(x) if isinstance(x, str) else x
    except Exception:
        vals = [v.strip() for v in str(x).split(",")]

    if not isinstance(vals, (list, tuple, set)):
        return []

    return [
        str(v).strip().strip("'\"")
        for v in vals
        if str(v).strip() and "carnitine" not in str(v).lower()
    ]

In [90]:
def majority_table_by_unit(df, unit):
    sub = df[df["Unit"] == unit].dropna(subset=["Group", MAJORITY_COL])
    records = []

    for _, row in sub.iterrows():
        for m in parse_majority_list(row[MAJORITY_COL]):
            records.append({"Group": row["Group"], "Molecule": m})

    if not records:
        return pd.DataFrame(), pd.DataFrame()

    long = pd.DataFrame(records)
    counts = pd.crosstab(long["Group"], long["Molecule"])
    counts = counts.reindex(present_order(long["Group"])).fillna(0)

    props = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0)
    return counts, props.fillna(0)

In [91]:
def stacked_bar_plot(props, unit, out_path):
    if props.empty:
        return

    cols = [c for c in MOLECULE_PALETTE if c in props.columns]
    cols += [c for c in props.columns if c not in MOLECULE_PALETTE]
    colors = [MOLECULE_PALETTE.get(c, None) for c in cols]

    ax = props[cols].plot(
        kind="bar", stacked=True, figsize=(7, 4.8),
        color=colors, edgecolor="black", linewidth=0.3
    )

    ax.set_title(f"LD majority molecules | {unit}")
    ax.set_xlabel("Group")
    ax.set_ylabel("Proportion")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
    ax.legend(title="Molecule", bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()

### Part 1 Nuclei and LD count

In [92]:
part1_dir = ROOT_DIR / "part1_counts"

for sub in ["anova_global_combo", "anova_by_unit", "tukey_kramer_by_unit", "plots"]:
    (part1_dir / sub).mkdir(parents=True, exist_ok=True)

count_outcomes = ['Droplet Count per Cell', 'Lipid Droplet Avg Area um2', 'Lipid Droplet Count']

In [93]:
for outcome in count_outcomes:
    global_anova = scipy_anova_group_unit_combo(df, outcome)

    out = part1_dir / "anova_global_combo" / f"{safe_name(outcome)}_anova_group_unit_combo.csv"
    global_anova.to_csv(out, index=False)

    display(global_anova)

Not enough data points for WD + veh | TAL_Cortex


,outcome,factor,F,p_value,n_groups,significant
0,Droplet Count per Cell,Group_Unit,2.826322,0.001665,14,True


Not enough data points for WD + veh | TAL_Cortex


,outcome,factor,F,p_value,n_groups,significant
0,Lipid Droplet Avg Area um2,Group_Unit,5.73697,7.815910e-08,14,True


Not enough data points for WD + veh | TAL_Cortex


,outcome,factor,F,p_value,n_groups,significant
0,Lipid Droplet Count,Group_Unit,6.470818,7.467170e-09,14,True


In [94]:
for outcome in count_outcomes:
    rows = []

    for unit in sorted(df["Unit"].dropna().unique()):
        F, p, n_groups = scipy_anova_group_within_unit(df, outcome, unit)
        rows.append({
            "outcome": outcome,
            "unit": unit,
            "F": F,
            "p_value": p,
            "n_groups": n_groups,
            "significant": pd.notna(p) and p < ALPHA
        })

    anova_unit = pd.DataFrame(rows)
    out = part1_dir / "anova_by_unit" / f"{safe_name(outcome)}_anova_by_unit.csv"
    anova_unit.to_csv(out, index=False)

    display(anova_unit)

Not enough data points for WD + veh: 1


,outcome,unit,F,p_value,n_groups,significant
0,Droplet Count per Cell,EarlyPT,1.973843,0.142639,4,False
1,Droplet Count per Cell,LatePT,1.821619,0.161800,4,False
2,Droplet Count per Cell,TAL_Cortex,4.452783,0.025147,3,True
3,Droplet Count per Cell,TAL_Medulla,6.671622,0.004966,3,True


Not enough data points for WD + veh: 1


,outcome,unit,F,p_value,n_groups,significant
0,Lipid Droplet Avg Area um2,EarlyPT,17.605832,0.000002,4,True
1,Lipid Droplet Avg Area um2,LatePT,15.023641,0.000002,4,True
2,Lipid Droplet Avg Area um2,TAL_Cortex,8.917473,0.001704,3,True
3,Lipid Droplet Avg Area um2,TAL_Medulla,0.241513,0.787325,3,False


Not enough data points for WD + veh: 1


,outcome,unit,F,p_value,n_groups,significant
0,Lipid Droplet Count,EarlyPT,0.269945,0.846467,4,False
1,Lipid Droplet Count,LatePT,6.077513,0.001994,4,True
2,Lipid Droplet Count,TAL_Cortex,0.792377,0.466476,3,False
3,Lipid Droplet Count,TAL_Medulla,6.666634,0.004982,3,True


In [95]:
for outcome in count_outcomes:
    anova_unit_path = part1_dir / "anova_by_unit" / f"{safe_name(outcome)}_anova_by_unit.csv"
    anova_unit = pd.read_csv(anova_unit_path)

    all_posthoc = []

    for _, row in anova_unit.iterrows():
        if bool(row["significant"]):
            post = scipy_posthoc_by_unit(df, outcome, row["unit"], equal_var=True)
            all_posthoc.append(post)

    if all_posthoc:
        posthoc_df = pd.concat(all_posthoc, ignore_index=True)
    else:
        posthoc_df = pd.DataFrame()

    out = part1_dir / "tukey_kramer_by_unit" / f"{safe_name(outcome)}_tukey_kramer.csv"
    posthoc_df.to_csv(out, index=False)

TypeError: tukey_hsd() got an unexpected keyword argument 'equal_var'

In [96]:
for outcome in count_outcomes:
    out_plot_dir = part1_dir / "plots" / safe_name(outcome)
    out_plot_dir.mkdir(parents=True, exist_ok=True)

    for unit in sorted(df["Unit"].dropna().unique()):
        df_filtered = df.groupby(['Unit', 'Group']).filter(lambda g: len(g) >= 3)
        df_filtered = df_filtered.groupby('Unit').filter(
            lambda g: g['Group'].nunique() >= len(df['Group'].unique())
            )

        out_path = out_plot_dir / f"{safe_name(unit)}.png"
        violin_scatter_plot(df_filtered, outcome, unit, out_path)

p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

WD + veh vs. WD + dapa: Custom statistical test, P_val:7.224e-04
NC + dapa vs. WD + dapa: Custom statistical test, P_val:1.124e-03
NC + veh vs. WD + dapa: Custom statistical test, P_val:2.252e-04
p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

NC + veh vs. NC + dapa: Custom statistical test, P_val:2.660e-02
NC + dapa vs. WD + dapa: Custom statistical test, P_val:2.448e-02
NC + veh vs. WD + veh: Custom statistical test, P_val:3.331e-02
NC + veh vs. WD + dapa: Custom statistical test, P_val:2.843e-03
p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.

### Part 2 Molecular Compositions

In [97]:
part2_dir = ROOT_DIR / "part2_molecules"
part2_dir.mkdir(parents=True, exist_ok=True)

In [98]:
for mol in MOLECULE_COLUMNS:
    mol_dir = part2_dir / safe_name(mol)

    for sub in ["anova_global_combo", "anova_by_unit", "games_howell_by_unit", "plots"]:
        (mol_dir / sub).mkdir(parents=True, exist_ok=True)

In [99]:
for mol in MOLECULE_COLUMNS:
    mol_dir = part2_dir / safe_name(mol)

    global_anova = scipy_anova_group_unit_combo(df, mol)

    out = mol_dir / "anova_global_combo" / f"{safe_name(mol)}_anova_group_unit_combo.csv"
    global_anova.to_csv(out, index=False)

    display(global_anova)

Not enough data points for WD + veh | TAL_Cortex


,outcome,factor,F,p_value,n_groups,significant
0,Sterols Normalized,Group_Unit,6.74838,1.497459e-07,11,True


Not enough data points for WD + veh | TAL_Cortex


,outcome,factor,F,p_value,n_groups,significant
0,Phospholipids Normalized,Group_Unit,3.05865,0.002438,11,True


Not enough data points for WD + veh | TAL_Cortex


...

,outcome,factor,F,p_value,n_groups,significant
0,Lipid Droplet Glycerols Normalized,Group_Unit,7.593729,2.157942e-08,11,True


Not enough data points for WD + veh | TAL_Cortex


,outcome,factor,F,p_value,n_groups,significant
0,Lipid Droplet Sphingolipids Normalized,Group_Unit,4.490354,0.000049,11,True


Not enough data points for WD + dapa | TAL_Cortex


,outcome,factor,F,p_value,n_groups,significant
0,Lipid Droplet Fatty Acids Normalized,Group_Unit,1.35941,0.239182,8,False


In [100]:
for mol in MOLECULE_COLUMNS:
    mol_dir = part2_dir / safe_name(mol)
    rows = []

    for unit in sorted(df["Unit"].dropna().unique()):
        F, p, n_groups = scipy_anova_group_within_unit(df, mol, unit)
        rows.append({
            "outcome": mol,
            "unit": unit,
            "F": F,
            "p_value": p,
            "n_groups": n_groups,
            "significant": pd.notna(p) and p < ALPHA
        })

    anova_unit = pd.DataFrame(rows)
    out = mol_dir / "anova_by_unit" / f"{safe_name(mol)}_anova_by_unit.csv"
    anova_unit.to_csv(out, index=False)

Not enough data points for WD + veh: 1
Not enough data points for WD + veh: 1
Not enough data points for WD + veh: 1
Not enough data points for WD + veh: 1
Not enough data points for WD + veh: 1
Not enough data points for WD + veh: 1
Not enough data points for WD + veh: 1
Not enough data points for WD + veh: 1
Not enough data points for WD + veh: 1
Not enough data points for WD + dapa: 1


In [102]:
for mol in MOLECULE_COLUMNS:
    mol_dir = part2_dir / safe_name(mol)
    anova_path = mol_dir / "anova_by_unit" / f"{safe_name(mol)}_anova_by_unit.csv"
    anova_unit = pd.read_csv(anova_path)

    for _, row in anova_unit.iterrows():
        unit = row["unit"]

        if bool(row["significant"]):
            gh = scipy_posthoc_by_unit(df, mol, unit, equal_var=False)
        else:
            gh = pd.DataFrame([{
                "outcome": mol,
                "unit": unit,
                "note": "Games-Howell not run; unit ANOVA not significant"
            }])

        out = mol_dir / "games_howell_by_unit" / f"{safe_name(unit)}_games_howell.csv"
        gh.to_csv(out, index=False)

TypeError: tukey_hsd() got an unexpected keyword argument 'equal_var'

In [103]:
for mol in MOLECULE_COLUMNS:
    mol_dir = part2_dir / safe_name(mol)

    for unit in sorted(df["Unit"].dropna().unique()):

        df_filtered = df.groupby(['Unit', 'Group']).filter(lambda g: len(g) >= 3)
        df_filtered = df_filtered.groupby('Unit').filter(
            lambda g: g['Group'].nunique() >= len(df['Group'].unique())
            )
        # display(df_filtered[["Unit", "Group", mol]].dropna().groupby("Unit").describe())
        print(f"Plotting {mol} | {unit}")
        out_path = mol_dir / "plots" / f"{safe_name(unit)}.png"
        violin_scatter_plot(df_filtered, mol, unit, out_path)

Plotting Sterols Normalized | EarlyPT
Plotting Sterols Normalized | LatePT
p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

NC + dapa vs. WD + veh: Custom statistical test, P_val:1.875e-03
WD + veh vs. WD + dapa: Custom statistical test, P_val:2.529e-03
NC + veh vs. WD + veh: Custom statistical test, P_val:1.483e-02
NC + veh vs. WD + dapa: Custom statistical test, P_val:1.887e-02
Plotting Sterols Normalized | TAL_Cortex
Plotting Sterols Normalized | TAL_Medulla
Plotting Phospholipids Normalized | EarlyPT
p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

NC + dapa vs. WD + veh: Custom statistical test, P_val:1.236e-02
WD + veh vs. WD + dapa: Custom statistical test, P_val:3.094e-03
NC + veh vs. WD + veh: Cu

### Part 3 LD Majority Molecules

In [104]:
part3_dir = ROOT_DIR / "part3_majority_molecules"

for sub in ["counts_tables", "proportion_tables", "plots"]:
    (part3_dir / sub).mkdir(parents=True, exist_ok=True)

In [105]:
for unit in sorted(df["Unit"].dropna().unique()):
    counts, props = majority_table_by_unit(df, unit)

    counts_path = part3_dir / "counts_tables" / f"{safe_name(unit)}_counts.csv"
    props_path = part3_dir / "proportion_tables" / f"{safe_name(unit)}_proportions.csv"

    counts.to_csv(counts_path)
    props.to_csv(props_path)

    out_plot = part3_dir / "plots" / f"{safe_name(unit)}_stacked_bar.png"
    stacked_bar_plot(props, unit, out_plot)

In [106]:
print(f"Done. All outputs saved under: {ROOT_DIR.resolve()}")

Done. All outputs saved under: C:\Users\jvill\Downloads\jorge_kidney_data_annotated_yang\jorge_kidney_data_annotated_yang
